# Generating text using GPT-2

```Thank you Tel Aviv. Great Tel Aviv, amazing Tel Aviv. This is an amazing exercise, tremendous exercise. The best exercise for the AI COE.```

```But I'm tired. I'm tired writing my speeches, great speeches, amazing speeches, amazing people. Create a model, tremendous model, to write my speech.```

```~Trump```

(Well it's actually Cordova...)

## Imports

In [1]:
# pytorch
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForCausalLM, AutoTokenizer
# Data
import numpy as np
import pandas as pd
# other
import os
from time import time

# seed
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
seed_everything(42)

C:\Users\NOAMCO3\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Generate text from a pre-trained model

```First, generate text using some great model, like GPT-2. Decode using beam search, Top-k and Top-p, great methods, amazing methods, great America.```

```I heard the radical democrats are using AutoTokenizer and AutoModelForCausalLM from transformers. They think they can win with this, they can't. They can't.```

In [2]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

In [3]:
# add the EOS token as PAD token to avoid warnings
model = AutoModelForCausalLM.from_pretrained("gpt2", pad_token_id=tokenizer.eos_token_id)

In [4]:
model_inputs = tokenizer('Look at me! I am Trump!', return_tensors='pt')

In [ ]:
# To see a solution for decode methods implementation, move to the part after fine-tuning

In [5]:
# Using the transformers library for generating text before fine-tuning.
new_tokens_num = 150
output = model.generate(**model_inputs, max_new_tokens=new_tokens_num, num_beams=5, no_repeat_ngram_size=4, early_stopping=True)
print("Beam search:")
print(tokenizer.decode(output[0], skip_special_tokens=True))

output = model.generate(**model_inputs, max_new_tokens=new_tokens_num, do_sample=True, top_k=15)
print("\n- - - - - - - - - - - - - - - - \nTop k:")
print(tokenizer.decode(output[0], skip_special_tokens=True))

output = model.generate(**model_inputs, max_new_tokens=new_tokens_num, do_sample=True, top_k=0, top_p=0.75)
print("\n- - - - - - - - - - - - - - - - \nTop p:")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Beam search:
Look at me! I am Trump! I am the president of the United States of America!"

Trump's campaign manager, Kellyanne Conway, told CNN's Wolf Blitzer on Sunday that she was "very disappointed" by Trump's comments.

"I'm very disappointed by what he's saying," Conway said. "I think he's going to have to be very careful about what he says. I think he should be very careful with what he says."

She added: "I think it's very important for him to be careful with his words. I think it's important for the American people to know what he's talking about."

In a statement, Trump's campaign said it was "deeply disappointed" by the remarks.

The


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



- - - - - - - - - - - - - - - - 
Top k:
Look at me! I am Trump! I am your father!"

Trump, like many of the Trump's onetime surrogates, seemed unaware of her father's record when he appeared on CNN on Sunday. He was the first foreign leader to speak out against the president.

"I have no doubt in my mind that the president has been a great friend of mine who I believe was a good friend of mine," Trump said, referring to a friend whose father died in Vietnam. "I have been a good friend of mine, and I will continue to be so."

The president's former foreign policy adviser, retired Lt. Gen. H.R. McMaster, was also interviewed by CNN.

McMaster, who said he was told about Trump's remarks

- - - - - - - - - - - - - - - - 
Top p:
Look at me! I am Trump! I am Donald Trump! I am the real Donald Trump!"

They were interrupted by the mayor's son, who continued the chants, "Allahu Akbar!" at a rally in downtown Berkeley.

[Video: Trump pours into 'Obama hating' crowds]

White House press secreta

## Read the data

```Now create a dataset with my speeches from ```https://www.kaggle.com/datasets/christianlillelund/donald-trumps-rallies.
```, amazing speeches, great speeches, great writing. Thank you, letters, you're heroes.```

```Decide a size for the input sequence length, and divide my amazing speeches to sequences of this size. The democrats are using 128, but we're at least double of them! We're double in size! Double in patriotism! Great double.```

In [36]:
trump_speech_full = []
for cnt, file in enumerate(os.listdir("Trump_speeches")):
    if file == ".ipynb_checkpoints":
        continue
    with open("Trump_speeches//" + file, "r", encoding="utf8") as f:
        trump_speech_full.append(f.read())

trump_speech = []
cnt = 0
size = 256
max_len = size
for cur_spch in trump_speech_full:
    txt_tok = tokenizer.tokenize(cur_spch)
    for s in range(0, len(txt_tok) - size + 1, size):
        trump_speech.append([cnt, tokenizer.convert_tokens_to_string(txt_tok[s: s + size])])
        cnt += 1

trump_speech = pd.DataFrame(trump_speech, columns=["id", "text"])
trump_speech["text"] += tokenizer.eos_token

len(trump_speech_full), trump_speech.shape

(35, (1816, 2))

## Tokenize

```Use the tremendous method encode, to encode the great data. Only the USA has such data.```

In [37]:
%%time
speeches = []
for sen in trump_speech["text"]:
    enc = tokenizer.encode(sen, add_special_tokens=True, padding="max_length", max_length=max_len, return_tensors = 'pt')
    speeches.append(enc)
speeches = torch.concat(speeches)
speeches.shape

CPU times: total: 1.5 s
Wall time: 1.53 s


torch.Size([1816, 257])

In [38]:
train_speech = TensorDataset(speeches)

## Fine-tuning


```Now, fine-tune your amazing model with the amazing dataset, and generate my next great speech. It should be a tremendous speech, amazing speech, hero speech. A speech you cry from, if you're not a man, men don't cry, only democrats.```

In [458]:
BATCH_SIZE = 128
SAMPLES_NUM = speeches.shape[0]
EPOCHS = 3
LEARNING_RATE = 1e-4
MAX_SEQ_LEN = max_len

In [459]:
model.train()
opt = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
proc_seq_count = 0
sum_loss = 0.0
batch_count = 0
num_of_sen = 0

In [460]:
train_dataloader = DataLoader(train_speech, batch_size=1, shuffle=True)

In [461]:
%%time
for epoch in range(EPOCHS):
    num_of_sen = 0
    start_time = time()
    print(f"EPOCH {epoch} started" + '=' * 30)
    for idx, sen in enumerate(train_dataloader):
        # if num_of_sen < 0: # change number to last checkpoint - **VERY** bad practice....
        #     num_of_sen += 1
        #     continue
        sen = sen[0]
        outputs = model(sen, labels=sen)
        loss, logits = outputs[:2]
        loss.backward()
        sum_loss = sum_loss + loss.detach().data

        proc_seq_count = proc_seq_count + 1
        if proc_seq_count == BATCH_SIZE or idx == SAMPLES_NUM - 1:
            proc_seq_count = 0
            batch_count += 1
            opt.step()
            opt.zero_grad()
            model.zero_grad()

        num_of_sen += 1
        print(f"\rsentence num: {num_of_sen}", end="")
        if batch_count == 1:
            print(f"\rsentence num: {num_of_sen}, sum loss: {sum_loss:.5f}, time: {time()-start_time:.2f}")
            batch_count = 0
            sum_loss = 0.0
            torch.save(model.state_dict(), "after_fine_tune_gen_txt.pt")
    print()

EPOCH 0 started==============================
sentence num: 128, sum loss: 338.98264, time: 298.64
sentence num: 256, sum loss: 331.54468, time: 570.14
sentence num: 384, sum loss: 344.97156, time: 832.88
sentence num: 512, sum loss: 331.37543, time: 1077.48
sentence num: 640, sum loss: 333.54227, time: 1316.28
sentence num: 768, sum loss: 328.86807, time: 1552.65
sentence num: 896, sum loss: 337.47568, time: 1791.77
sentence num: 1024, sum loss: 331.70422, time: 2043.12
sentence num: 1152, sum loss: 329.09164, time: 2312.98
sentence num: 1280, sum loss: 332.26035, time: 2572.51
sentence num: 1408, sum loss: 333.19257, time: 2814.32
sentence num: 1536, sum loss: 335.19574, time: 3056.99
sentence num: 1664, sum loss: 330.84201, time: 3317.08
sentence num: 1792, sum loss: 331.00351, time: 3561.63
sentence num: 1816EPOCH 1 started==============================
sentence num: 1920, sum loss: 327.17273, time: 241.23
sentence num: 2048, sum loss: 325.65201, time: 493.68
sentence num: 2176, su

## Self implementation of the decoding methods

In [464]:
org_txt = 'Cows should be'

In [465]:
# Beam search
model.eval()
beams_num = 5
with torch.no_grad():
    txt = org_txt
    model_inputs = tokenizer(txt, return_tensors='pt')["input_ids"]
    prob = torch.Tensor([1])
    txt_lst = [txt]
    input_lst = [model_inputs]
    new_tokens_num = 30
    
    for i in range(new_tokens_num):
        next_poss_txt = []
        next_poss_prb = []
        next_poss_input = []
        for beam in range(len(input_lst)):
            output = model(input_lst[beam])
            next_prob = nn.functional.softmax(output.logits[0][-1], dim=0)
            tops = next_prob.topk(beams_num)
            for opt in range(beams_num):
                next_poss_prb.append(prob[beam] * tops[0][opt])
                next_poss_input.append(torch.concat([input_lst[beam], tops[1][opt].reshape(1, 1)], dim=1))
                next_poss_txt.append(txt_lst[beam] + tokenizer.decode(tops[1][opt], skip_special_tokens=True))
        next_poss_prb = torch.Tensor(next_poss_prb)
        indc = next_poss_prb.topk(beams_num)[1]
        
        txt_lst = [next_poss_txt[i] for i in indc]
        prob = next_poss_prb[indc]
        prob = prob / prob.min()
        input_lst = [next_poss_input[i] for i in indc]
    for i in txt_lst:
        print(i)
        print()

Cows should be proud of themselves. They should be proud of themselves. They should be proud of themselves. They should be proud of themselves. They should be proud of

Cows should be proud of themselves. They should be proud of their families. They should be proud of their country. They should be proud of themselves. They should be

Cows should be proud of themselves. They should be proud of their families. They should be proud of their country. They should be proud of their country. They should

Cows should be proud of themselves. They should be proud of themselves. They should be proud of themselves. They should be proud of themselves. They should be proud.

Cows should be proud of themselves. They should be proud of their families. They should be proud of their communities. They should be proud of their country. They should



In [466]:
# Top k
model.eval()
beams_num = 5
with torch.no_grad():
    txt = org_txt
    model_inputs = tokenizer(txt, return_tensors='pt')["input_ids"]
    new_tokens_num = 30
    top_k = 20
    
    for i in range(new_tokens_num):
        output = model(model_inputs)
        next_prob = nn.functional.softmax(output.logits[0][-1], dim=0)
        tops = next_prob.topk(top_k)
        p = tops[0].numpy()
        p = p / p.sum()
        a = tops[1].numpy()
        next_token = int(np.random.choice(a, p=p))

        model_inputs = torch.concat([model_inputs, torch.Tensor([next_token]).reshape(1, 1)], dim=1).int()
        txt = txt + tokenizer.decode(next_token, skip_special_tokens=True)
        
    print(txt)

Cows should be allowed to farm, you know that? No. They should be allowed to do what they love. They should have them, and they should have everything


In [467]:
# Top p
model.eval()
beams_num = 5
with torch.no_grad():
    txt = org_txt
    model_inputs = tokenizer(txt, return_tensors='pt')["input_ids"]
    new_tokens_num = 30
    top_p = 0.8
    
    for i in range(new_tokens_num):
        output = model(model_inputs)
        next_prob = nn.functional.softmax(output.logits[0][-1], dim=0)
        for k in range(50257):
            tops = next_prob.topk(k)
            p = tops[0].numpy()
            if p.sum() < top_p:
                continue
            p = p / p.sum()
            a = tops[1].numpy()
            break
        next_token = int(np.random.choice(a, p=p))

        model_inputs = torch.concat([model_inputs, torch.Tensor([next_token]).reshape(1, 1)], dim=1).int()
        txt = txt + tokenizer.decode(next_token, skip_special_tokens=True)
        
    print(txt)

Cows should be like this, they should be so good. They should be so good. They're doing a fantastic job and it's a shame, but they're


## Using the transformers library implementation

In [468]:
model.eval()
with torch.no_grad():
    model_inputs = tokenizer('New York is the city of', return_tensors='pt')
    new_tokens_num = 300
    output = model.generate(**model_inputs, max_new_tokens=new_tokens_num, num_beams=8, no_repeat_ngram_size=4, early_stopping=False)

    print("Beam search:")
    print(tokenizer.decode(output[0], skip_special_tokens=True))

    output = model.generate(**model_inputs, max_new_tokens=new_tokens_num, do_sample=True, top_k=30)
    print("\n\n- - - - - - - - - - - - - - - - \nTop k:")
    print(tokenizer.decode(output[0], skip_special_tokens=True))

    output = model.generate(**model_inputs, max_new_tokens=new_tokens_num, do_sample=True, top_k=0, top_p=0.75)
    print("\n\n- - - - - - - - - - - - - - - - \nTop p:")
    print(tokenizer.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Beam search:
New York is the city of New York. It's the most beautiful city in the world. It's one of the most beautiful places in the world, and it's a great place to be. It's a beautiful place to be in New York. And you know what? We're going to win New York, and we're going to keep it that way. We're not going to let it get away from us. We are going to win the great state of New York and we are going to keep America out of endless foreign wars, endless wars, endless foreign wars. And we will never stop fighting for the values that bind us together as one America. We support, protect, and defend the Constitution of the United States. We stand with the incredible heroes of law enforcement. We believe in the dignity of work and the sanctity of life. We believe that faith and family, not government and bureaucracy, are the true American way. And we believe that children should be taught to love our country, honor our history, and always respect our great American flag. And we live by t

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.




- - - - - - - - - - - - - - - - 
Top k:
New York is the city of the New York Yankees. We love you. We love you, and we're great friends with you all. We're working again with the Yankees. It's a great city. It's great to be here. We love you, and we're going to work hard. And to give you relief, the U.S. Senate has passed a record number of Republican bills expanding opportunities for school choice, early education, early release and other opportunities. And in particular, we need the Supreme Court. So I wanted to introduce you on that because I'll be announcing to our great friends in the Senate that we're going to win back the Senate so badly with the Democrats. But we need this. The Democrats are a disaster. So I'm going to introduce the Senate and there are a lot of Democrats, and I was watching the debates last night, one by one, the numbers came back up. And I said, "That guy is a genius, but you can't run for reelection. I can't run because my father took care of me and he too

In [462]:
model.eval()
with torch.no_grad():
    model_inputs = tokenizer('I think that cows should', return_tensors='pt')
    new_tokens_num = 300
    output = model.generate(**model_inputs, max_new_tokens=new_tokens_num, num_beams=8, no_repeat_ngram_size=4, early_stopping=False)

    print("Beam search:")
    print(tokenizer.decode(output[0], skip_special_tokens=True))

    output = model.generate(**model_inputs, max_new_tokens=new_tokens_num, do_sample=True, top_k=30)
    print("\n\n- - - - - - - - - - - - - - - - \nTop k:")
    print(tokenizer.decode(output[0], skip_special_tokens=True))

    output = model.generate(**model_inputs, max_new_tokens=new_tokens_num, do_sample=True, top_k=0, top_p=0.75)
    print("\n\n- - - - - - - - - - - - - - - - \nTop p:")
    print(tokenizer.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Beam search:
I think that cows should be allowed to graze freely in the open air, right? Right? Right? They should be able to graze. They shouldn't be forced to do that. They should be free to do whatever they want to do. They should have the right to do what they want. They should not be forced to go out and graze. You know why? Because they don't want to hurt the cows. They don't want the cows to be hurt. They want them to be free to graze, and that's what we're doing. We're protecting cows, and we're protecting your right to keep and bear arms. You know what that is? That's what it is. It's called the Second Amendment, right? It's the Second Amendment. It's the most important thing you can have. You can keep your family, you can keep your loved ones, and you can protect your Second Amendment. We will never stop fighting for the sacred values that bind us together as one America. We support, protect and defend the Constitution of the United States. We stand with the incredible heroes

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.




- - - - - - - - - - - - - - - - 
Top k:
I think that cows should be allowed to graze on the surface of the ocean. In the extreme right, we're bringing in the Environmental Protection Agency. We've taken in a record amount of environmental regulations since I took office. You see what's happening. You see what's happening. They've done a good job, so we need to get them all cleaned up. We need to do something. But you know what, it's true. It's great to see. We have a lot of enthusiasm for the Environmental Protection Agency, and now they're going to get rid. They're going to get rid of some regulations, but if you look at my record as president, over four years, I did a good job, but I didn't have the energy. It was easy being president, but I had the energy, and I was a little surprised that I didn't get the energy. I was surprised to hear that, actually, but the last four years, we've put an increase of $2.3 billion, but for the last four years, the total amount of foreign investme

In [463]:
model.eval()
with torch.no_grad():
    model_inputs = tokenizer('Taylor Swift is', return_tensors='pt')
    new_tokens_num = 300
    output = model.generate(**model_inputs, max_new_tokens=new_tokens_num, num_beams=5, no_repeat_ngram_size=3, early_stopping=False)

    print("Beam search:")
    print(tokenizer.decode(output[0], skip_special_tokens=True))

    output = model.generate(**model_inputs, max_new_tokens=new_tokens_num, do_sample=True, top_k=30)
    print("\n- - - - - - - - - - - - - - - - \nTop k:")
    print(tokenizer.decode(output[0], skip_special_tokens=True))

    output = model.generate(**model_inputs, max_new_tokens=new_tokens_num, do_sample=True, top_k=0, top_p=0.8)
    print("\n- - - - - - - - - - - - - - - - \nTop p:")
    print(tokenizer.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Beam search:
Taylor Swift is doing a great job. I don't know if you've heard of her, but she's doing a fantastic job. She's a great friend of mine, and I'm thrilled to be working with her. Thank you very much. And I want to thank all of you for being with us tonight. We have a lot of work to do, but we're going to have a great time tonight. And we're also joined tonight by a man who's been with us for many, many years, and he's done an incredible job. And by the way, he's a big fan of mine. He's been a big supporter of mine for a long time. He was with us all the way from New Hampshire. He said, "You know, I'm a big believer in the sanctity of life. I believe that faith and family, not government and bureaucracy, are the true American way. We believe that children should be taught to love our country, honor our history, and always respect our great American flag. We stand on the shoulders of American heroes who crossed the oceans, settled the continent, tamed the wilderness, won two Wo

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



- - - - - - - - - - - - - - - - 
Top k:
Taylor Swift is a talented speaker. But you know, I think she's going to be great. But that's okay. That might be the most important. But she's a great leader on one, two, three issues, two big issues in the state of Minnesota. That's a big issue for the Republican party and we have tremendous support. We have some big endorsements, and we've got a great group of people that are really, really talented. But Nancy, I really want to thank you. I'd love to have her on because she's been incredible. I just got off the plane and she said, "You better sign that sheet of paper." I signed it. And I was like, "No, I can't. Nobody's signed it." But we need some good talent and we need a team that understands a lot about these things. But, thank you very much. So thank you. So

- - - - - - - - - - - - - - - - 
Top p:
Taylor Swift is right about that. But you know what? They're trying to hurt our economy. They're trying to hurt our country. They're trying t